# Extended Analysis

Runs the extended benchmark comparison and the allocation + tree heuristic ablation on the saved extended results.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Resolve paths whether the notebook is launched from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
MODULE_DIRECTORY = PROJECT_DIRECTORY
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

# Reload algorithm code first so comparison_analysis sees the latest heuristic signature.
import heuristic
heuristic = importlib.reload(heuristic)

import approximate_comparison_analysis as comparison_analysis
comparison_analysis = importlib.reload(comparison_analysis)
approximate_summary = comparison_analysis.approximate_summary
field_family_summary = comparison_analysis.field_family_summary
load_approximate_results = comparison_analysis.load_approximate_results
overall_error_curve_summary = comparison_analysis.overall_error_curve_summary
runtime_by_k_summary = comparison_analysis.runtime_by_k_summary
run_approximate_dataset = comparison_analysis.run_approximate_dataset

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)


## Run Settings

The `RUN_SCOPE` setting selects either the deterministic 100-case pilot sample (`"test"`) or all 2,700 expanded cases per dimension (`"full"`). Both scopes use the same generated benchmark instances; the setting only changes how many saved cases are selected for evaluation.


In [ ]:
# Run the expanded one-dimensional and two-dimensional datasets.
DATASETS = ["benchmark_1d", "benchmark_2d"]

# Switch here: "test" runs 100 balanced cases; "full" runs all 2,700 cases per dataset.
RUN_SCOPE = "test"
if RUN_SCOPE not in {"test", "full"}:
    raise ValueError('RUN_SCOPE must be "test" or "full"')

REPLICATES_PER_CELL = None
TEST_CASES_PER_DATASET = 100
CASE_LIMIT = None if RUN_SCOPE == "full" else TEST_CASES_PER_DATASET
CASE_SAMPLE_SEED = 20260820
PROGRESS_EVERY = 25

# Keep scalable results separate from the compact exact comparison.
RESULTS_ROOT = MODULE_DIRECTORY / "results" / "cmp_extended"

print("Run scope:", RUN_SCOPE)
print("Cases per dataset:", "2,700" if RUN_SCOPE == "full" else TEST_CASES_PER_DATASET)
print("Test-case sampling seed:", CASE_SAMPLE_SEED)
print("Results directory:", RESULTS_ROOT)


## Algorithm Configuration



In [ ]:
# Greedy has no optional tuning parameters.
GREEDY_PARAMETERS = {}

# Configure allocation VND and final beam-tree construction.
HEURISTIC_PARAMETERS = {
    "beam_width": 3,
    "initialization": "near_uniform",
    "move_sizes": (1, 2, 3),
}

# Reproduce the near-uniform remainder placement.
UNIFORM_PARAMETERS = {"random_seed": 0}

# Reproduce case-specific random allocations and decision trees.
RANDOM_PARAMETERS = {"random_seed": 0}

print("Greedy parameters:", GREEDY_PARAMETERS)
print("Heuristic parameters:", HEURISTIC_PARAMETERS)
print("Uniform parameters:", UNIFORM_PARAMETERS)


In [ ]:
# Reload the runner so an open kernel cannot reuse an older case selector.
comparison_analysis = importlib.reload(comparison_analysis)
run_approximate_dataset = comparison_analysis.run_approximate_dataset

records_by_dataset = {}
for dataset in DATASETS:
    display(Markdown(f"### Running `{dataset}`"))
    records_by_dataset[dataset] = run_approximate_dataset(
        dataset,
        replicates_per_cell=REPLICATES_PER_CELL,
        results_root=RESULTS_ROOT,
        greedy_parameters=GREEDY_PARAMETERS,
        heuristic_parameters=HEURISTIC_PARAMETERS,
        uniform_parameters=UNIFORM_PARAMETERS,
        random_parameters=RANDOM_PARAMETERS,
        progress_every=PROGRESS_EVERY,
        case_limit=CASE_LIMIT,
        case_sample_seed=CASE_SAMPLE_SEED,
    )


## Numerical Summary

This table summarizes surrogate final error and runtime for each scalable method. It is a compact check on the full set of records used by the subsequent plots.


In [ ]:
# Load saved records when the expensive run cell was intentionally skipped.
if "records_by_dataset" not in globals():
    records_by_dataset = {
        dataset: load_approximate_results(
            dataset, results_root=RESULTS_ROOT, run_scope=RUN_SCOPE
        )
        for dataset in DATASETS
    }

summary_lines = [
    "| Dataset | Algorithm | Cases | Budget runs | Mean surrogate error | Maximum surrogate error | Mean runtime (s) | Median runtime (s) |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for dataset, records in records_by_dataset.items():
    for row in approximate_summary(records):
        summary_lines.append(
            f"| {dataset} | {row['algorithm']} | {row['cases']} | {row['runs']} | "
            f"{row['mean_error']:.5f} | {row['maximum_error']:.5f} | "
            f"{row['mean_seconds']:.5f} | {row['median_seconds']:.5f} |"
        )

display(Markdown("\n".join(summary_lines)))


## Overall Surrogate-Error Curve Summary

Each row matches one point in the overall mean surrogate-error curves. The final column reports the surrogate-error reduction achieved by the Allocation + tree heuristic relative to Greedy; positive values mean the Allocation + tree heuristic is better.


In [ ]:
curve_summary_lines = [
    r"| Dimension | Budget level | Mean $N/N_{\min}^{opt}$ | Greedy mean surrogate error | Heuristic mean surrogate error | Uniform mean surrogate error | Random mean surrogate error | Heuristic reduction from Greedy |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in overall_error_curve_summary(records):
        curve_summary_lines.append(
            f"| {row['dimension']} | {row['budget_level']} | "
            f"{row['mean_budget_fraction']:.3f} | "
            f"{row['greedy_mean_error']:.4f} | "
            f"{row['heuristic_mean_error']:.4f} | "
            f"{row['uniform_mean_error']:.4f} | "
            f"{row['random_mean_error']:.4f} | "
            f"{row['error_reduction']:.4f} "
            f"({row['error_reduction_percent']:.1f}%) |"
        )

display(Markdown("\n".join(curve_summary_lines)))


## Runtime Summary by Number of Fields



In [ ]:
runtime_summary_lines = [
    "| Dimension | $K$ | Greedy median runtime (s) | Heuristic median runtime (s) | Uniform median runtime (s) | Random median runtime (s) | Heuristic / Greedy runtime |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in runtime_by_k_summary(records):
        runtime_summary_lines.append(
            f"| {row['dimension']} | {row['K']} | "
            f"{row['greedy_median_seconds']:.6f} | "
            f"{row['heuristic_median_seconds']:.6f} | "
            f"{row['uniform_median_seconds']:.6f} | "
            f"{row['random_median_seconds']:.6f} | "
            f"{row['heuristic_to_greedy_ratio']:.2f}x |"
        )

display(Markdown("\n".join(runtime_summary_lines)))


## Surrogate-Error Summary by Field Family

Each row averages surrogate final error over all included $K$, $M$, sensor budgets, and replicates. The final column reports the Allocation + tree heuristic's surrogate-error reduction relative to Greedy.


In [ ]:
family_summary_lines = [
    "| Dimension | Field family | Greedy mean surrogate error | Heuristic mean surrogate error | Uniform mean surrogate error | Random mean surrogate error | Heuristic reduction from Greedy |",
    "|---|---|---:|---:|---:|---:|---:|",
]
for records in records_by_dataset.values():
    for row in field_family_summary(records):
        family_summary_lines.append(
            f"| {row['dimension']} | {row['field_family']} | "
            f"{row['greedy_mean_error']:.4f} | "
            f"{row['heuristic_mean_error']:.4f} | "
            f"{row['uniform_mean_error']:.4f} | "
            f"{row['random_mean_error']:.4f} | "
            f"{row['heuristic_reduction']:.4f} "
            f"({row['heuristic_reduction_percent']:.1f}%) |"
        )

display(Markdown("\n".join(family_summary_lines)))


## Comparison Plots

These figures report surrogate final error because this notebook evaluates the optimization objective used by the algorithms.

The detailed mean-error figure retains separate panels for each $K$. A second summarized figure shows 1D and 2D panels averaged across all included $K$, $M$, and field families. The field-family figure uses vertical boxplots and the same muted algorithm colors. The heuristic-iteration figure shows the distribution of iterations performed before stopping, with the configured maximum shown by a dashed line.


In [ ]:
# Close figures left in memory by an earlier notebook run.
plt.close("all")

# Reload and apply the shared style before creating new figures.
import plot_style
plot_style = importlib.reload(plot_style)
plot_style.configure_plot_style()
# Reload plotting code so an open kernel does not reuse an older function.
comparison_analysis = importlib.reload(comparison_analysis)
# Load saved records when the expensive run cell was intentionally skipped.
if "records_by_dataset" not in globals():
    records_by_dataset = {
        dataset: comparison_analysis.load_approximate_results(
            dataset, results_root=RESULTS_ROOT, run_scope=RUN_SCOPE
        )
        for dataset in DATASETS
    }
figures = comparison_analysis.make_combined_approximate_plots(
    records_by_dataset,
    results_root=RESULTS_ROOT,
    run_scope=RUN_SCOPE,
)
for title, figure in figures:
    display(Markdown(f"### {title}"))
    display(figure)
    plt.close(figure)


## Saved Files

```text
results/cmp_extended/
├── extended_1d_<test|full>/tables/
├── extended_2d_<test|full>/tables/
└── summary_extended_1d_extended_2d_<test|full>/figures/


## Heuristic Ablation

This section compares four saved strategies on the extended datasets: Greedy, Uniform, Uniform with the VND allocation from the heuristic but a simple tree, and the full allocation + tree heuristic. The comparison separates the contribution of allocation refinement from the final beam-search tree.


In [ ]:
import csv
from pathlib import Path
from statistics import fmean
import importlib
import sys

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

# Resolve paths whether the notebook is launched from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import approximate_comparison_analysis
import heuristic
import input_data
import plot_style
import uniform

approximate_comparison_analysis = importlib.reload(approximate_comparison_analysis)
heuristic = importlib.reload(heuristic)
input_data = importlib.reload(input_data)
plot_style = importlib.reload(plot_style)
uniform = importlib.reload(uniform)
plot_style.configure_plot_style()

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)


## Settings

The ablation uses the same `RUN_SCOPE` convention as the extended comparison. The `"test"` scope is the deterministic pilot sample; the `"full"` scope uses the complete regenerated extended results.

In [ ]:
RUN_SCOPE = "test"
DATASETS = ("benchmark_1d", "benchmark_2d")
FINAL_TREE_BEAM_WIDTH = 3
MOVE_SIZES = (1, 2, 3)

RESULTS_ROOT = PROJECT_DIRECTORY / "results" / "cmp_extended"
INPUT_ROOT = PROJECT_DIRECTORY / "inputs"
SUMMARY_DIRECTORY = RESULTS_ROOT / f"summary_heuristic_ablation_{RUN_SCOPE}"
TABLES_DIRECTORY = SUMMARY_DIRECTORY / "tables"
FIGURES_DIRECTORY = SUMMARY_DIRECTORY / "figures"
SAVE_OUTPUTS = True

print("Run scope:", RUN_SCOPE)
print("Extended results root:", RESULTS_ROOT)
print("Input root:", INPUT_ROOT)
print("Final tree beam width:", FINAL_TREE_BEAM_WIDTH)
print("Allocation move sizes:", MOVE_SIZES)
print("Save outputs:", SAVE_OUTPUTS)


In [ ]:
METHOD_ORDER = ("Greedy", "Uniform", "Uniform + VND", "Full Heuristic")
METHOD_STYLES = {
    "Greedy": {"color": plot_style.METHOD_COLORS["Greedy"], "linestyle": plot_style.METHOD_LINESTYLES["Greedy"], "marker": plot_style.METHOD_MARKERS["Greedy"]},
    "Uniform": {"color": plot_style.METHOD_COLORS["Uniform"], "linestyle": plot_style.METHOD_LINESTYLES["Uniform"], "marker": plot_style.METHOD_MARKERS["Uniform"]},
    "Uniform + VND": {"color": plot_style.METHOD_COLORS["Optimal"], "linestyle": plot_style.METHOD_LINESTYLES["Optimal"], "marker": plot_style.METHOD_MARKERS["Optimal"]},
    "Full Heuristic": {"color": plot_style.METHOD_COLORS["Heuristic"], "linestyle": plot_style.METHOD_LINESTYLES["Heuristic"], "marker": plot_style.METHOD_MARKERS["Heuristic"]},
}


def load_extended_records(dataset):
    return approximate_comparison_analysis.load_approximate_results(
        dataset,
        results_root=RESULTS_ROOT,
        run_scope=RUN_SCOPE,
    )


def vnd_simple_tree_error(record):
    K = int(record["K"])
    M = int(record["M"])
    pairs = tuple((i, j) for i in range(K) for j in range(i + 1, K))
    _, _, _, pair_error_table = input_data.load_sparse_input(
        INPUT_ROOT / str(record["dataset"]) / f"{record['case']}.csv"
    )
    allocation = tuple(int(value) for value in record["heuristic_allocation"])
    pair_locations, pair_errors = heuristic._compute_pair_errors(
        pairs,
        M,
        allocation,
        pair_error_table,
    )
    tree = uniform._build_tree(tuple(range(K)), pair_locations, pair_errors)
    return uniform._tree_error(tree, K)


def build_ablation_records(records_by_dataset):
    ablation_records = []
    for dataset, records in records_by_dataset.items():
        display(Markdown(f"### Reconstructing VND-only baseline for `{dataset}`"))
        for index, record in enumerate(records, start=1):
            vnd_only_error = vnd_simple_tree_error(record)
            ablation_records.append({
                "dataset": dataset,
                "case": str(record["case"]),
                "field_model": str(record["field_model"]),
                "dimension": int(record["spatial_dimension"]),
                "K": int(record["K"]),
                "M": int(record["M"]),
                "budget_level": int(record["budget_level"]),
                "budget_ratio": float(record["budget_fraction_of_N_min"]),
                "Greedy": float(record["greedy_error"]),
                "Uniform": float(record["uniform_error"]),
                "Uniform + VND": vnd_only_error,
                "Full Heuristic": float(record["heuristic_error"]),
            })
            if index == 1 or index % 100 == 0 or index == len(records):
                print(f"{dataset}: reconstructed {index}/{len(records)}")
    return ablation_records


def mean_by_budget(records):
    rows = []
    for dimension in sorted({record["dimension"] for record in records}):
        dimension_records = [record for record in records if record["dimension"] == dimension]
        for budget_level in sorted({record["budget_level"] for record in dimension_records}):
            selected = [record for record in dimension_records if record["budget_level"] == budget_level]
            row = {
                "dimension": f"{dimension}D",
                "budget_level": budget_level,
                "budget_ratio": fmean(record["budget_ratio"] for record in selected),
                "runs": len(selected),
            }
            for method in METHOD_ORDER:
                row[method] = fmean(record[method] for record in selected)
            row["vnd_gain_vs_uniform"] = row["Uniform"] - row["Uniform + VND"]
            row["full_gain_vs_greedy"] = row["Greedy"] - row["Full Heuristic"]
            row["full_gain_vs_uniform"] = row["Uniform"] - row["Full Heuristic"]
            row["beam_gain_after_vnd"] = row["Uniform + VND"] - row["Full Heuristic"]
            rows.append(row)
    return rows


def overall_summary(records):
    rows = []
    for dimension in sorted({record["dimension"] for record in records}):
        selected = [record for record in records if record["dimension"] == dimension]
        row = {
            "dimension": f"{dimension}D",
            "runs": len(selected),
        }
        for method in METHOD_ORDER:
            row[method] = fmean(record[method] for record in selected)
        row["vnd_gain_vs_uniform"] = row["Uniform"] - row["Uniform + VND"]
        row["full_gain_vs_greedy"] = row["Greedy"] - row["Full Heuristic"]
        row["full_gain_vs_uniform"] = row["Uniform"] - row["Full Heuristic"]
        row["beam_gain_after_vnd"] = row["Uniform + VND"] - row["Full Heuristic"]
        rows.append(row)
    return rows


def markdown_table(rows, columns, formats):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---:" if key in formats else "---" for _, key in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            formatter = formats.get(key)
            values.append(formatter(row[key]) if formatter else str(row[key]))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])


def write_rows_csv(path, rows):
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
records_by_dataset = {dataset: load_extended_records(dataset) for dataset in DATASETS}
ablation_records = build_ablation_records(records_by_dataset)

summary_rows = overall_summary(ablation_records)
budget_rows = mean_by_budget(ablation_records)

if SAVE_OUTPUTS:
    write_rows_csv(TABLES_DIRECTORY / "overall_summary.csv", summary_rows)
    write_rows_csv(TABLES_DIRECTORY / "budget_summary.csv", budget_rows)
    print("Saved ablation tables under:", TABLES_DIRECTORY)

display(Markdown("### Overall Ablation Summary"))
display(Markdown(markdown_table(
    summary_rows,
    (
        ("Dimension", "dimension"),
        ("Runs", "runs"),
        ("Greedy", "Greedy"),
        ("Uniform", "Uniform"),
        ("Uniform + VND", "Uniform + VND"),
        ("Full Heuristic", "Full Heuristic"),
        ("VND gain vs Uniform", "vnd_gain_vs_uniform"),
        ("Full gain vs Greedy", "full_gain_vs_greedy"),
        ("Beam gain after VND", "beam_gain_after_vnd"),
    ),
    {
        "runs": lambda value: f"{int(value)}",
        "Greedy": lambda value: f"{value:.5f}",
        "Uniform": lambda value: f"{value:.5f}",
        "Uniform + VND": lambda value: f"{value:.5f}",
        "Full Heuristic": lambda value: f"{value:.5f}",
        "vnd_gain_vs_uniform": lambda value: f"{value:.5f}",
        "full_gain_vs_greedy": lambda value: f"{value:.5f}",
        "beam_gain_after_vnd": lambda value: f"{value:.5f}",
    },
)))

display(Markdown("### Ablation Summary by Budget"))
display(Markdown(markdown_table(
    budget_rows,
    (
        ("Dimension", "dimension"),
        ("Budget level", "budget_level"),
        ("Budget ratio", "budget_ratio"),
        ("Runs", "runs"),
        ("Greedy", "Greedy"),
        ("Uniform", "Uniform"),
        ("Uniform + VND", "Uniform + VND"),
        ("Full Heuristic", "Full Heuristic"),
        ("VND gain vs Uniform", "vnd_gain_vs_uniform"),
        ("Full gain vs Greedy", "full_gain_vs_greedy"),
        ("Beam gain after VND", "beam_gain_after_vnd"),
    ),
    {
        "budget_level": lambda value: f"{int(value)}",
        "budget_ratio": lambda value: f"{value:.3f}",
        "runs": lambda value: f"{int(value)}",
        "Greedy": lambda value: f"{value:.5f}",
        "Uniform": lambda value: f"{value:.5f}",
        "Uniform + VND": lambda value: f"{value:.5f}",
        "Full Heuristic": lambda value: f"{value:.5f}",
        "vnd_gain_vs_uniform": lambda value: f"{value:.5f}",
        "full_gain_vs_greedy": lambda value: f"{value:.5f}",
        "beam_gain_after_vnd": lambda value: f"{value:.5f}",
    },
)))


In [ ]:
def plot_ablation_curves(rows):
    dimensions = sorted({row["dimension"] for row in rows})
    figure, axes = plt.subplots(1, len(dimensions), figsize=(10.8, 4.0), sharey=True)
    if len(dimensions) == 1:
        axes = [axes]
    for axis, dimension in zip(axes, dimensions):
        selected = [row for row in rows if row["dimension"] == dimension]
        for method in METHOD_ORDER:
            style = METHOD_STYLES[method]
            axis.plot(
                [row["budget_ratio"] for row in selected],
                [row[method] for row in selected],
                color=style["color"],
                linestyle=style["linestyle"],
                marker=style["marker"],
                linewidth=1.8,
                markersize=5.0,
                label=method,
            )
        axis.set_title(f"{dimension} extended data set")
        axis.set_xlabel(r"Budget ratio $\rho=N/N_{\min}$")
        axis.grid(alpha=0.22, linewidth=0.8)
        axis.set_axisbelow(True)
    axes[0].set_ylabel("Mean surrogate misclassification probability")
    handles, labels = axes[-1].get_legend_handles_labels()
    figure.legend(handles, labels, loc="outside upper center", ncol=len(labels))
    figure.tight_layout(rect=(0, 0, 1, 0.88))
    return figure


figure = plot_ablation_curves(budget_rows)

if SAVE_OUTPUTS:
    FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)
    figure.savefig(FIGURES_DIRECTORY / "heuristic_ablation_extended.png", dpi=600, bbox_inches="tight")
    figure.savefig(FIGURES_DIRECTORY / "heuristic_ablation_extended.pdf", bbox_inches="tight")
    print("Saved ablation figures under:", FIGURES_DIRECTORY)

display(figure)
plt.close(figure)


## Saved Files

When `SAVE_OUTPUTS = True`, this notebook writes:

```text
results/cmp_extended/summary_heuristic_ablation_<test|full>/
├── tables/
│   ├── overall_summary.csv
│   └── budget_summary.csv
└── figures/
    ├── heuristic_ablation_extended.pdf
    └── heuristic_ablation_extended.png
```
